# Target valuations

This notebook is dedicated to the basic math behind the target valuations I have in mind.

## Loan payment calculations

In [12]:
def monthly_payment(principal: float, annual_interest_rate: float, years: int) -> float:
    """
    Calculate the monthly payment for an amortized simple-interest loan.s

    :param principal: The amount of the loan
    :param annual_interest_rate: The annual interest rate (as a decimal)
    :param years: The number of years to pay off the loan
    :return: The monthly payment amount
    """
    monthly_interest_rate = annual_interest_rate / 12
    number_of_payments = years * 12

    ratio = monthly_interest_rate * (1 + monthly_interest_rate) ** number_of_payments / ((1 + monthly_interest_rate) ** number_of_payments - 1)
    return principal * ratio

def monthly_sba_payment(pricipal: float, annual_interest_rate: float) -> float:
    """
    Calculate the monthly payment for a simple interest SBA loan, amortized over 10 years.
    """
    return monthly_payment(pricipal, annual_interest_rate, 10)  # considering 10 year loans

def annual_sba_payment(principal: float, annual_interest_rate: float) -> float:
    """
    Calculate the annual payment for a simple interest SBA loan, amortized over 10 years.
    """
    return monthly_sba_payment(principal, annual_interest_rate) * 12

In [13]:
print("Test values at $100k:")
print(f"monthly: {monthly_sba_payment(100_000, 0.1)}\nannual: {annual_sba_payment(100_000, 0.1)}")

Test values at $100k:
monthly: 1321.5073688176192
annual: 15858.088425811431


## Conservative cost estimates

I'm going to go off some basic Gemini ideas for realistic closing costs and then consider working capital scenarios

### Closing costs

It's estimated SBA guaranty fee and legal or other banker fees will add 3%-6% of the business price to the project costs.

### Working capital

I'm currently going on a gut feeling of $50,000 in necessary day-1 working capital, but I think a range of $30,000-$75,000 is realistic for a conservative analysis.

## Deal structure

### Down payment

I would ideally want to negotiate a 5% full standby (life of the loan) seller note as part of the initial 10% equity injection.  Either way, this hurts my personal liquidity and not the post-close cash flow.

### Interest rates

I think 10% is reasonable (high!), but should consider up to 12% to be safe.

## Putting it all together: SDE and price estimates

In [20]:
def annual_debt_cost(price: float, equity_injection: float, annual_interest_rate: float, closing_ratio: float=0.06, working_capital: float=50_000) -> float:
    """
    Calculate the all-in cost of a loan, including down payment and working capital.

    :param price: The total price of the item being financed
    :param equity_injection: The amount not financed (includes full standby note)
    :param annual_interest_rate: The annual interest rate (as a decimal)
    :param closing_ratio: The ratio of closing costs to the loan amount (default is 0.06)
    :param working_capital: Additional working capital required (default is $50,000)
    :return: The total all-in cost of the loan
    """
    loan_amount = price - equity_injection + working_capital
    closing_costs = price * closing_ratio
    total_cost = loan_amount + closing_costs

    if equity_injection < 0.1 * total_cost:
        print(f"Warning: Equity injection is less than 10% of total cost. Consider increasing equity injection to at least ${0.1 * total_cost:.2f}.")

    return annual_sba_payment(total_cost, annual_interest_rate)

def annual_cash_flow(price: float, cash_flow: float, equity_injection: float, annual_interest_rate: float, closing_ratio: float=0.06, working_capital: float=50_000) -> float:
    """
    Calculate the annual cash flow required to service the debt.

    :param price: The total price of the item being financed
    :param equity_injection: The amount not financed (includes full standby note)
    :param annual_interest_rate: The annual interest rate (as a decimal)
    :param closing_ratio: The ratio of closing costs to the loan amount (default is 0.06)
    :param working_capital: Additional working capital required (default is $50,000)
    :return: The annual cash flow required to service the debt
    """
    return cash_flow - annual_debt_cost(price, equity_injection, annual_interest_rate, closing_ratio, working_capital)

In [21]:
annual_debt_cost(price=350_000, equity_injection=40_000, annual_interest_rate=0.1, closing_ratio=0.03, working_capital=50_000)

58754.21761763135

In [27]:
annual_cash_flow(price=350_000, cash_flow=120_000, equity_injection=40_000, annual_interest_rate=0.1, closing_ratio=0.03, working_capital=50_000)

61245.78238236865